In [51]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from scipy.stats import randint

In [52]:
df = pd.read_csv("/content/diabetes_prediction_dataset.csv")

In [53]:
df.head()

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


In [54]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   gender               100000 non-null  object 
 1   age                  100000 non-null  float64
 2   hypertension         100000 non-null  int64  
 3   heart_disease        100000 non-null  int64  
 4   smoking_history      100000 non-null  object 
 5   bmi                  100000 non-null  float64
 6   HbA1c_level          100000 non-null  float64
 7   blood_glucose_level  100000 non-null  int64  
 8   diabetes             100000 non-null  int64  
dtypes: float64(3), int64(4), object(2)
memory usage: 6.9+ MB


In [55]:
df.diabetes.value_counts()

,count
diabetes,
0,91500
1,8500


In [56]:
df.isnull().sum()

,0
gender,0
age,0
hypertension,0
heart_disease,0
smoking_history,0
bmi,0
HbA1c_level,0
blood_glucose_level,0
diabetes,0


In [57]:
X = df.drop("diabetes", axis=1)
y = df["diabetes"]

In [58]:
X = pd.get_dummies(X, drop_first=True)

In [59]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [60]:
print(X_train.shape)
print(X_test.shape)

(80000, 13)
(20000, 13)


In [61]:
# Decision Tree

model = DecisionTreeClassifier(random_state=42)

cv_scores_baseline = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
baseline_cv_acc = cv_scores_baseline.mean()

model.fit(X_train, y_train)
baseline_test_acc = accuracy_score(y_test, model.predict(X_test))

In [62]:
param_grid = {
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

In [63]:
# GridSearchCV
grid_search = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Extract best metrics
grid_best_params = grid_search.best_params_
grid_cv_acc = grid_search.best_score_
grid_test_acc = accuracy_score(y_test, grid_search.best_estimator_.predict(X_test))

In [64]:
param_dist = {
    'max_depth': [None, 3, 5, 10, 15, 20],
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 20),
    'criterion': ['gini', 'entropy']
}

In [65]:
# RandomizedSearchCV (50 random combinations)

random_search = RandomizedSearchCV(DecisionTreeClassifier(random_state=42), param_distributions=param_dist, n_iter=50, cv=5, scoring='accuracy', random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

# Extract best metrics
random_best_params = random_search.best_params_
random_cv_acc = random_search.best_score_
random_test_acc = accuracy_score(y_test, random_search.best_estimator_.predict(X_test))

In [66]:
# Random Forest

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

rf_cv_scores = cross_val_score(rf_model, X_train, y_train, cv=5, scoring='accuracy')
rf_cv_acc = rf_cv_scores.mean()

rf_model.fit(X_train, y_train)
rf_test_acc = accuracy_score(y_test, rf_model.predict(X_test))

In [67]:
# Saving Parameters

rf_params = {'n_estimators': 100, 'random_state': 42}

In [68]:
# Results

results = [
    {
        "Model": "Baseline Decision Tree",
        "CV Accuracy": round(baseline_cv_acc, 4),
        "Test Accuracy": round(baseline_test_acc, 4),
        "Best Params": "Default parameters"
    },
    {
        "Model": "GridSearchCV (Tree)",
        "CV Accuracy": round(grid_cv_acc, 4),
        "Test Accuracy": round(grid_test_acc, 4),
        "Best Params": str(grid_best_params)
    },
    {
        "Model": "RandomizedSearchCV (Tree)",
        "CV Accuracy": round(random_cv_acc, 4),
        "Test Accuracy": round(random_test_acc, 4),
        "Best Params": str(random_best_params)
    },
    {
        "Model": "Random Forest (Ensemble)",
        "CV Accuracy": round(rf_cv_acc, 4),
        "Test Accuracy": round(rf_test_acc, 4),
        "Best Params": str(rf_params)
    }
]

# Create DataFrame and display
results_df = pd.DataFrame(results)
results_df.set_index("Model", inplace=True)
display(results_df)

,CV Accuracy,Test Accuracy,Best Params
Model,,,
Baseline Decision Tree,0.9515,0.9513,Default parameters
GridSearchCV (Tree),0.9718,0.9723,"{'criterion': 'gini', 'max_depth': 5, 'min_sam..."
RandomizedSearchCV (Tree),0.9718,0.9723,"{'criterion': 'entropy', 'max_depth': 5, 'min_..."
Random Forest (Ensemble),0.9697,0.9703,"{'n_estimators': 100, 'random_state': 42}"
